# SetHolder — Set Holder Client

Client-side agent.  Builds a local Bloom filter, RSS3-shares it to
the three servers.  Uses reserve/connect callbacks — **no HTTP
dependency** in the mpmt package itself.


In [ ]:
import mpmt

def my_reserve(token, action):
    return [{"port": 9001}, {"port": 9002}, {"port": 9003}]

def my_connect(token, action, leader_port, helper_a_port, helper_b_port):
    pass

sh = mpmt.SetHolder(
    set_size=2**10,
    fpr_mantissa=1.0,
    fpr_exponent=-3,
    reserve_fn=my_reserve,
    connect_fn=my_connect,
)
print(f"bf_size={sh.bf_size}, hf_num={sh.hf_num}")
print(f"token={sh.token.hex()}")


## Protocol Flow

### join / update

1. ``sh._reserve(token, action)`` → 3 ports from the servers
2. ``sh._connect(token, action, leader_port, helper_a_port, helper_b_port)``
3. Create temp Rep3 ring with Helpers via ``_build_rep3_channels``
4. Receive hash seeds from Leader via RingTransport
5. Build local BF: ``hash_aes_dm`` → ``ring_mod`` → ``batch_set``
6. ``share_vector`` → ``send_vector`` (this_share + nxt_share to Leader)

### quit

Reserve only — no connect phase needed.


## Pre-allocated Buffers

- ``sh.bf_buf`` — ``Rvector(1)(bf_size)`` (local BF)
- ``sh.sv_buf`` — ``ShrRep3ShareVec(1)(bf_size)`` (RSS3 shares)
- ``sh.aux_buf`` — ``RvectorPack(1)(bf_size)`` (scratch)
